In [1]:
# Text splitter functionality is provided by LangChain framework
from langchain_text_splitters import HTMLHeaderTextSplitter, RecursiveCharacterTextSplitter

# Make use of BS for hadling the web content
import requests
from bs4 import BeautifulSoup

import lancedb

# Sentence transformers to use the embedding models locally
from sentence_transformers import SentenceTransformer, util
import pandas as pd

#### Utilities

> Library functions

**Custom Meta Data**  
From the parsed and split content, this function helps to create meta data in a custom way, that can be used while creating Knowledge DB

In [2]:
def meta_data_from_headings (heading: dict, n: int = 1, from_end: bool = True, sep: str = " : ") -> str:
    """
    Concatenates n values from a heading dictionary, either from the start or from the end.

    Param:
        heading (dict): Input dictionary for headings.
        n (int): Number of elements to take.
        from_end (bool): If True, take from the end; else from the start.
        sep (str): Optional separator to use between concatenated strings.

    Returns: Meta data as concatenation of headings.
    """
    values = list(heading.values())

    if n <= 0:
        n = 1
    if n > len(values):
        n = len(values)

    # Select n items from start or end
    selected = values[-n:] if from_end else values[:n]

    # Always concatenate in forward direction
    return sep.join(str(v) for v in selected)

**Get Main Content**

In [3]:
def get_main_content (url, type):

    html = requests.get(url).text
    soup = BeautifulSoup(html, "html.parser")

    # Remove layout elements
    for tag in soup(["nav", "header", "footer", "aside", "script", "style"]):
        tag.decompose()

    # Check and get main section of the pages
    main = soup.find("main")

    if not main:
        
        # fallback method, if no 'main' section in html page
        candidates = soup.find_all("div", recursive=True)
        main = max(candidates, key=lambda c: len(c.get_text(strip=True)), default=soup.body)

    # Get cleaned HTML content. Tags retained
    main_html = str(main)

    # If HTML content is required, provide with the tags
    if type == 'html':
        return (main_html)

    # If text is requirred, provide only the text content
    elif type == 'text':

        text_soup = BeautifulSoup (main_html, "html.parser")
        main_text = text_soup.get_text(separator="\n", strip=True)
        return main_text

**Multi-Pass Chunking**
> Often the scenario could be to incorporate multiple ways of chunking to have better granularity and meaning in the chunks  
> The Sentence and content aware chunkers are used in Combination to retain the context and be granular as well  
> The context is captured by the Meta data

In [4]:
# Define what are the splitters to be considered. There is default in library itself
seperators = [".", "?", "!"]

# Splitter function based on seperator and the length criteria
text_splitter = RecursiveCharacterTextSplitter (chunk_size=300, chunk_overlap=0,
                                                length_function=len, is_separator_regex=False,
                                                keep_separator=False,
                                                separators=seperators,
                                                )

# levels of header tags in html to split on
header_levels = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
    ("h4", "Header 4"),
]

# Define a Splitter object for HTML content from the lib
# This library also gives splitter for Markdown, JSON etc
html_splitter = HTMLHeaderTextSplitter(header_levels)

**Combine 2 methods**  
Get the content and split it based on document structure first  
Some of the chunks can be big, because of the way the text is present  
Pass those blocks for one more level of splitting by sentences  
Capture the meta data from the headings and use it along with the text

In [ ]:
def Build_Chunks (url, source, chunk_size_limit):

    # Get the main content
    HTML_Content = get_main_content (url, "html")

    # Chunk based on document structure
    docs = html_splitter.split_text (HTML_Content)

    # Start with empty list
    Chunks = []

    with open ('chunks.txt', mode='w') as f:

        for doc in docs :

            try :

                meta_data = meta_data_from_headings (doc.metadata)

                if not meta_data:
                    meta_data = 'Generic'

                # If the chunk is too long,
                if (len(doc.page_content) > chunk_size_limit):

                    # Split by sentece(s) by shorter lenth
                    splits = text_splitter.split_text(doc.page_content)

                    # Make them individual chunk with same meta data
                    for split in splits:

                        # Capture if the meta data and text are not the same
                        if (meta_data != split):

                            Chunk = {'source': source,'topic' : meta_data, 'text' : split}
                            print (Chunk, "\n----",file=f)

                            Chunks = Chunks + [Chunk]
                        
                else :
                    
                    if (meta_data != doc.page_content):
                        
                        Chunk = {'source': source, 'topic' : meta_data, 'text' : doc.page_content}
                        print (Chunk, "\n----",file=f)
                        Chunks = Chunks + [Chunk]
                
                # print (doc.metadata)
                # print ("Content : ", doc.page_content,"\n---")
                
            except Exception :
                pass

    print (len(Chunks))

    return Chunks

: 

In [18]:
url = "https://www.ibm.com/think/topics/cloud-computing"

Chunks = Build_Chunks (url, "IBM", 500)
Chunks

133


[{'source': 'IBM',
  'topic': 'Authors',
  'text': 'Stephanie  Susnjara  \nStaff Writer  \nIBM Think  \nIan Smalley  \nStaff Editor  \nIBM Think'},
 {'source': 'IBM',
  'topic': 'What is cloud computing?',
  'text': 'Cloud computing is on-demand access to computing resources—physical or virtual servers, data storage, capabilities, application development tools, software, AI-powered analytic platforms and more—over the internet with pay-per-use pricing.  \nnetworking  \nThink Newsletter'},
 {'source': 'IBM',
  'topic': 'Join over 100,000 subscribers who read the latest news in tech',
  'text': 'Stay up to date on the most important—and intriguing—industry trends on AI, automation, data and beyond with the Think newsletter.\xa0See the .  \nIBM Privacy Statement'},
 {'source': 'IBM',
  'topic': 'What is cloud computing?',
  'text': 'Your subscription will be delivered in English. You will find an unsubscribe link in every newsletter. You can manage your subscriptions or unsubscribe . Refe

In [ ]:
Chunks

**Vectorise the data**  
> Once the chunks are creared along with the supporging data, use embedding model and craete vectors  
> Use a HF model which is suitable for general purpose 

In [19]:
Embedder = SentenceTransformer ("sentence-transformers/all-MiniLM-L6-v2")

In [20]:
# Create vectors and store in the Chunks 
for idx, Chunk in enumerate (Chunks):

    vector = Embedder.encode (Chunk['text'])
    # print (type(vector))
    # print (vector)

    Chunks[idx]['vector'] = vector.tolist ()
  

In [21]:
Chunks

[{'source': 'IBM',
  'topic': 'Authors',
  'text': 'Stephanie  Susnjara  \nStaff Writer  \nIBM Think  \nIan Smalley  \nStaff Editor  \nIBM Think',
  'vector': [-0.031386882066726685,
   -0.03286300599575043,
   -0.005354499910026789,
   -0.01620291918516159,
   0.018706955015659332,
   0.03935916721820831,
   -0.025970513001084328,
   0.10353484749794006,
   0.09397298842668533,
   0.04360019415616989,
   0.03803868964314461,
   0.07019742578268051,
   -0.045608144253492355,
   -0.051652852445840836,
   0.014489779248833656,
   0.03755968436598778,
   -0.01681271567940712,
   -0.04828549921512604,
   -0.05528660863637924,
   -0.04510766640305519,
   -0.08553463220596313,
   0.015512513928115368,
   -0.03496822714805603,
   -0.021502088755369186,
   0.04763539507985115,
   -0.02646510675549507,
   -0.010443700477480888,
   -0.027140377089381218,
   -0.04878630116581917,
   0.05965946987271309,
   -0.03800581023097038,
   -0.0034169298596680164,
   0.019608397036790848,
   0.043435007333

In [10]:
# Check various topics existing in all Chunks
Topics = list({c["topic"] for c in Chunks})
Topics

['IaaS (Infrastructure-as-a-Service)',
 'Private cloud',
 'Generic',
 'Enable business continuity and\xa0disaster recovery',
 'Cloud use cases',
 'The modern hybrid multicloud',
 'Achieving AI-readiness with hybrid cloud',
 'Support edge and IoT environments',
 'Origins of cloud computing',
 'PaaS (Platform-as-a-Service)',
 'Serverless computing',
 'Build and test cloud-native applications',
 'Benefits of cloud computing',
 'Cloud computing components',
 'Authors',
 'SaaS (Software-as-a-Service)',
 'Cloud security management tools',
 'Scale infrastructure',
 'Multicloud',
 'Cloud security',
 'What is cloud computing?',
 'Join over 100,000 subscribers who read the latest news in tech',
 'Hybrid cloud',
 'Cloud computing services',
 'Use cutting-edge technologies',
 'Public cloud',
 'Cloud sustainability']

In [11]:
# Create a Lance DB Vector Base
DB = lancedb.connect ('Vector_DB')

# Create a Table and add the Chunks data
table = DB.create_table("article", data=Chunks, mode="overwrite") 
print (table.schema)

source: string
topic: string
text: string
vector: fixed_size_list<item: float>[384]
  child 0, item: float


In [12]:
# Query a vector
Query = "Platforms used as business in today's world"

Query_Vector = Embedder.encode (Query).tolist ()

Results = table.search(Query_Vector).limit(5).to_list ()

for Rs in Results :

    print (Rs['_distance']," ## ",Rs ['text'])

1.1794707775115967  ##  Platform as a service (PaaS)  
With PaaS the cloud provider hosts everything at their data center. These include servers, networks, storage, operating system software, and databases
1.2251909971237183  ##  business sustainability  
All major cloud players have made net-zero commitments to reduce their carbon footprints and help clients reduce the energy they typically consume using an on-premises setup. For instance, IBM is driven by initiatives to reach NetZero by 2030.  
sustainable procurement
1.2362241744995117  ##  For example, an organization can choose AWS for its global reach with web hosting, IBM Cloud for data analytics and platforms and Microsoft Azure for its security features
1.264312982559204  ##  provides software developers with an on-demand platform—hardware, complete software stack, infrastructure and development tools—for running, developing and managing applications without the cost, complexity and inflexibility of maintaining that platform o

**Create a Tech Repo**  
> From various sources in internet, create a knowledge repo with all information chunked and vectorised

In [23]:
# Gather multiple reference material for Technology information
References = [{'Source' : 'Microsoft', 'url' : "https://azure.microsoft.com/en-us/resources/cloud-computing-dictionary/what-is-cloud-computing"},
              {'Source' : 'IBM', 'url' : "https://www.ibm.com/think/topics/cloud-computing"},
              {'Source' : 'Oracle', 'url' : "https://www.oracle.com/in/cloud/what-is-cloud-computing/"},
              {'Source' : 'AWS', 'url' : "https://aws.amazon.com/what-is/iot/"},
              {'Source' : 'IBM', 'url' : "https://www.ibm.com/think/topics/edge-ai"},              
              {'Source' : 'Microsoft', 'url' : "https://azure.microsoft.com/en-us/resources/cloud-computing-dictionary/what-is-edge-computing"},
              {'Source' : 'IBM', 'url' : "https://www.ibm.com/think/topics/edge-computing"},              
              {'Source' : 'Fortinet', 'url' : "https://www.fortinet.com/resources/cyberglossary/edge-computing"},
              {'Source' : 'NVIDIA', 'url' : "https://blogs.nvidia.com/blog/what-is-edge-ai/"},
              {'Source' : 'MIT', 'url' : "https://mitsloan.mit.edu/ideas-made-to-matter/machine-learning-explained"},
              {'Source' : 'AWS', 'url' : "https://aws.amazon.com/what-is/machine-learning/"}
            ]

Chunks = []
for Ref in References:
    
    Parts = Build_Chunks (Ref['url'], Ref ['Source'], 500)
    Chunks = Chunks + Parts

print (len(Chunks))

39
133
134
44
75
76
39
83
45
89
91
848


In [24]:
Chunks

[{'source': 'Microsoft',
  'topic': 'Generic',
  'text': 'A beginner’s guide  \nGet started with Azure'},
 {'source': 'Microsoft',
  'topic': 'Generic',
  'text': 'Cloud computing is the delivery of computing services—including servers, storage, databases, networking, software, analytics, and intelligence—over the internet (“the cloud”) to offer faster innovation, flexible resources, and economies of scale. You typically pay only for cloud services you use, helping you lower your operating costs, run your infrastructure more efficiently, and scale as your business needs change.'},
 {'source': 'Microsoft',
  'topic': 'Types of cloud computing',
  'text': 'Discover the top benefits of cloud computing on cost, performance, security, and more.  \nUnderstand the differences between public, private, and hybrid cloud deployment models.  \nLearn about the four broad categories of cloud services: IaaS, PaaS, serverless, and SaaS.  \nExplore cloud computing services from Microsoft—and find out w

In [14]:
Embedder_1 = SentenceTransformer ("sentence-transformers/all-mpnet-base-v2")

In [15]:
# Create vectors and store in the Chunks 
for idx, Chunk in enumerate (Chunks):

    vector = Embedder_1.encode (Chunk['text'])
    Chunks[idx]['vector'] = vector.tolist ()

In [16]:
# Create a Table and add the Chunks data
table = DB.create_table("tech_ref", data=Chunks, mode="overwrite") 
print (table.schema)

source: string
topic: string
text: string
vector: fixed_size_list<item: float>[768]
  child 0, item: float


In [17]:
# Query a vector
Query = "There are many service providers"

Query_Vector = Embedder_1.encode (Query).tolist ()

# Results = table.search(Query_Vector).distance_type("cosine").limit(5).to_list ()
Results = table.search().where("topic IN ('What is edge computing?')").to_list ()

for Rs in Results :

    # print (Rs['_distance'],Rs['source']," ## ",Rs ['text'])
    print (Rs['source']," ## ",Rs ['text'])

IBM  ##  Edge computing is a distributed computing framework that brings enterprise applications closer to data sources such as IoT devices or local edge servers
IBM  ##  This proximity to data at its source can deliver strong business benefits, including faster insights, improved response times and better bandwidth availability.  
The explosive growth and increasing computing power of IoT devices has resulted in massive volumes of data
IBM  ##  And data volumes continue to grow as 5G networks increase the number of connected mobile devices.  
In the past, the promise of cloud and AI was to automate and speed up innovation by driving actionable insight from data
IBM  ##  But connected devices create data at an extraordinary scale and complexity, outpacing network and infrastructure capabilities.  
Sending all device-generated data to a centralized data center or to the cloud causes bandwidth and latency issues
IBM  ##  Edge computing offers a more efficient alternative—data is processe